# Test the Chinook SQL Agent Connection

This notebook checks the pieces used by `sql_agent.py`: environment loading, OpenAI model access, MySQL connectivity, direct SQL queries, and a final natural-language SQL agent call.

## 1. Load Environment Variables

The notebook expects a repository-level `.env` file. The MySQL values can be omitted if your local defaults are `root`, no password, `localhost`, port `3306`, and database `Chinook`.

In [ ]:
from pathlib import Path
import os

from dotenv import load_dotenv

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / ".env").exists() or (path / "requirements.in").exists():
            return path
    raise RuntimeError("Could not find the repository root from the current notebook directory.")


project_root = find_repo_root(Path.cwd().resolve())
load_dotenv(project_root / ".env")
print("Project root:", project_root)

print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("MYSQL_HOST:", os.getenv("MYSQL_HOST", "localhost"))
print("MYSQL_DATABASE:", os.getenv("MYSQL_DATABASE", "Chinook"))

## 2. Test the OpenAI Chat Model

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"),
    temperature=float(os.getenv("OPENAI_TEMPERATURE", "0")),
    max_tokens=int(os.getenv("OPENAI_MAX_TOKENS", "512")),
)

llm.invoke("Reply with only: model connection ok").content

## 3. Connect to MySQL with LangChain

In [ ]:
from urllib.parse import quote_plus

from langchain_community.utilities.sql_database import SQLDatabase

mysql_username = os.getenv("MYSQL_USERNAME", "root")
mysql_password = os.getenv("MYSQL_PASSWORD", "")
mysql_host = os.getenv("MYSQL_HOST", "localhost")
mysql_port = os.getenv("MYSQL_PORT", "3306")
database_name = os.getenv("MYSQL_DATABASE", "Chinook")

mysql_uri = (
    f"mysql+mysqlconnector://{quote_plus(mysql_username)}:{quote_plus(mysql_password)}"
    f"@{mysql_host}:{mysql_port}/{database_name}"
)

db = SQLDatabase.from_uri(mysql_uri)
db.dialect, db.get_usable_table_names()[:5]

## 4. Run Direct SQL Checks

In [ ]:
print(db.run("SELECT COUNT(*) AS album_count FROM Album;"))
print(db.run("SELECT Name FROM Artist ORDER BY ArtistId LIMIT 5;"))

## 5. Create and Test the SQL Agent

In [ ]:
from langchain_classic.agents import AgentType
from langchain_community.agent_toolkits import create_sql_agent

agent_executor = create_sql_agent(
    llm=llm,
    db=db,
    verbose=True,
    handle_parsing_errors=True,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
)

result = agent_executor.invoke({"input": "How many albums are in the database?"})
result["output"]